In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==============================================================================
# PHASE 8: SYSTEM IMPORTS & ENVIRONMENT SETUP
# ==============================================================================
import os
import time
import gc
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from google.colab import drive
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Mount Google Drive
drive.mount('/content/drive')

# Device Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Global Hyperparameters
CONFIG = {
    "batch_size": 32,
    "epochs": 50,
    "lr": 5e-6,
    "num_classes": 19,
    "patience": 8,
    "label_smoothing": 0.1,
    "weight_decay": 1e-2
}

# Project Paths
BASE_DIR = "/content/drive/MyDrive/BDMediHerb/Original Dataset"
PROJECT_ROOT = "/content/drive/MyDrive/BDMediHerb/Outputs"
#BEST_3_NAMES = ["MobileNet_V3_Large", "ResNet50", "ViT_B16"]
BEST_3_NAMES = ["MobileNet_V3_Large", "ResNet50"]
#BEST_3_NAMES = ["ResNet50", "ViT_B16"]


# ==============================================================================
# PHASE 1: UTILITY CLASSES & FUNCTIONS
# ==============================================================================

class MapDataset(torch.utils.data.Dataset):
    """Wraps a subset to apply transforms dynamically."""
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.dataset[index]
        if self.transform: x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.dataset)

def initialize_architecture(name):
    """Initializes backbone architectures to EXACTLY match the saved weights."""
    if name == "MobileNet_V3_Large":
        model = models.mobilenet_v3_large(weights="DEFAULT")
        num_ftrs = model.classifier[3].in_features
        # This matches your original training Phase 3 structure
        model.classifier[3] = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, CONFIG["num_classes"])
        )
    elif name == "ResNet50":
        model = models.resnet50(weights="DEFAULT")
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, CONFIG["num_classes"])
    elif name == "ViT_B16":
        model = models.vit_b_16(weights="DEFAULT")
        num_ftrs = model.heads.head.in_features
        model.heads.head = nn.Linear(num_ftrs, CONFIG["num_classes"])
    return model

class FocalLoss(nn.Module):
    """Focal Loss to handle hard-to-classify samples at terminal accuracy stages."""
    def __init__(self, alpha=1, gamma=2, label_smoothing=0.05):
        super(FocalLoss, self).__init__()
        self.alpha = alpha; self.gamma = gamma; self.label_smoothing = label_smoothing
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

def run_failure_analysis(model, loader, class_names, export_path):
    """Exports misclassified images and error logs for research audit."""
    model.eval()
    error_dir = os.path.join(export_path, "Failure_Analysis")
    os.makedirs(error_dir, exist_ok=True)
    error_log = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            preds = outputs.max(1)[1].cpu()
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    error_log.append({
                        "Actual": class_names[labels[i]],
                        "Predicted": class_names[preds[i]]
                    })

    # Save the log to the export_path
    pd.DataFrame(error_log).to_csv(os.path.join(export_path, "misclassifications.csv"), index=False)

    # FIX: Changed export_dir to error_dir
    print(f"[INFO] Failure analysis report saved to {error_dir}")

# ==============================================================================
# PHASE 2: DATA RE-LOADING & STRATIFIED PARTITIONING
# ==============================================================================

print("\n" + "-"*30 + "\n[INFO] Loading Dataset and Partitioning...\n" + "-"*30)
hybrid_ds_raw = datasets.ImageFolder(BASE_DIR)
hybrid_labels = np.array(hybrid_ds_raw.targets)
hybrid_indices = np.arange(len(hybrid_labels))

# Stratified Split 70:15:15
idx_train_val_hybrid, hybrid_test_idx = train_test_split(
    hybrid_indices, test_size=0.15, stratify=hybrid_labels, random_state=42
)
idx_train_hybrid, hybrid_val_idx = train_test_split(
    idx_train_val_hybrid, test_size=(0.15/0.85), stratify=hybrid_labels[idx_train_val_hybrid], random_state=42
)

def get_hybrid_loaders(train_idx, val_idx, test_idx, img_size, full_ds):
    """Generates resolution-aware loaders with heavy training augmentations."""
    train_trans = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    eval_trans = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return {
        'train': DataLoader(MapDataset(Subset(full_ds, train_idx), train_trans), batch_size=CONFIG["batch_size"], shuffle=True),
        'val': DataLoader(MapDataset(Subset(full_ds, val_idx), eval_trans), batch_size=CONFIG["batch_size"], shuffle=False),
        'test': DataLoader(MapDataset(Subset(full_ds, test_idx), eval_trans), batch_size=CONFIG["batch_size"], shuffle=False)
    }

# Initialize Loaders
HYBRID_RES = 224
hybrid_loaders = get_hybrid_loaders(idx_train_hybrid, hybrid_val_idx, hybrid_test_idx, HYBRID_RES, hybrid_ds_raw)

# ==============================================================================
# PHASE 3: COMPONENT ARCHITECTURE & ENSEMBLE CORE
# ==============================================================================

class FeatureExtractor(nn.Module):
    """Wraps backbones and loads individual weights, stripping classification heads."""
    def __init__(self, model_name):
        super(FeatureExtractor, self).__init__()
        base_model = initialize_architecture(model_name)
        weights_path = os.path.join(PROJECT_ROOT, model_name, "best_model.pth")
        base_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))

        if "ViT" in model_name:
            self.feat_dim = base_model.heads.head.in_features
            base_model.heads = nn.Identity()
        elif "ResNet" in model_name:
            self.feat_dim = base_model.fc.in_features
            base_model.fc = nn.Identity()
        elif "MobileNet" in model_name:
            self.feat_dim = base_model.classifier[0].in_features
            base_model.classifier = nn.Identity()
        self.backbone = base_model

    def forward(self, x):
        x = self.backbone(x)
        if len(x.shape) > 2: x = torch.flatten(x, 1)
        return x

class EnsembleStackingNet(nn.Module):
    """Meta-learner that dynamically adapts to the number of input models."""
    def __init__(self, model_names, num_classes=19):
        super(EnsembleStackingNet, self).__init__()
        # Dynamic streams using ModuleList
        self.streams = nn.ModuleList([FeatureExtractor(name) for name in model_names])
        self.heads = nn.ModuleList([nn.Linear(stream.feat_dim, num_classes) for stream in self.streams])

        # Meta-learner: input size depends on the number of models
        self.meta_learner = nn.Sequential(
            nn.Linear(num_classes * len(model_names), 256),
            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        logits_list = []
        for i, stream in enumerate(self.streams):
            features = stream(x)
            logits = self.heads[i](features)
            logits_list.append(logits)

        stacked_logits = torch.cat(logits_list, dim=-1)
        return self.meta_learner(stacked_logits)


# ==============================================================================
# PHASE 4: TRAINING EXECUTION
# ==============================================================================

model_hybrid = EnsembleStackingNet(BEST_3_NAMES).to(DEVICE)
param_groups = []
for stream in model_hybrid.streams:
    param_groups.append({'params': stream.parameters(), 'lr': 1e-6})
for head in model_hybrid.heads:
    param_groups.append({'params': head.parameters(), 'lr': 1e-4})
param_groups.append({'params': model_hybrid.meta_learner.parameters(), 'lr': 1e-4})
optimizer = optim.AdamW(param_groups, weight_decay=0.01)
criterion = FocalLoss(gamma=2, label_smoothing=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)


# ------------------------------------------------------------------------------
# 5. EXECUTION: HIGH-PRECISION TRAINING & LATENCY TRACKING
# ------------------------------------------------------------------------------
best_hyb_acc = 0.0
early_stop_counter = 0
CONFIG["patience"] = 8

# Archive indices lengths for precise accuracy calculation
len_train = len(idx_train_hybrid)
len_val = len(hybrid_val_idx)

# Paths for saving
EXPORT_DIR_HYB = os.path.join(PROJECT_ROOT, "Hybrid_DualStreamMR")
os.makedirs(EXPORT_DIR_HYB, exist_ok=True)
local_path = "temp_best_hybrid.pth"
drive_path = os.path.join(EXPORT_DIR_HYB, "best_hybrid_model.pth")

# Initialize history dictionary before starting the loop
history_hyb = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'inf_ms': [], 'time_sec': [], 'lr': []}

print(f"\n{'Epoch':<8} | {'Tr. Loss':<10} | {'Tr. Acc':<10} | {'Val Loss':<10} | {'Val Acc':<10} | {'Inference':<12} | {'Time'}")
print("-" * 105)


# Now your existing loop starts...
for epoch in range(CONFIG["epochs"]):
    epoch_start = time.time()

    # --- TRAINING PHASE ---
    model_hybrid.train()
    tr_loss, tr_correct = 0.0, 0
    for imgs, lbls in hybrid_loaders['train']:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model_hybrid(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * imgs.size(0)
        tr_correct += (out.max(1)[1] == lbls).sum().item()

    # --- VALIDATION PHASE ---
    model_hybrid.eval()
    v_loss, v_correct, latencies = 0.0, 0, []
    with torch.no_grad():
        for imgs, lbls in hybrid_loaders['val']:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            tic = time.time()
            out = model_hybrid(imgs)
            latencies.append((time.time() - tic) / imgs.size(0))
            v_loss += criterion(out, lbls).item() * imgs.size(0)
            v_correct += (out.max(1)[1] == lbls).sum().item()

    # --- METRICS CALCULATION ---
    cur_t_acc = round(tr_correct / len_train, 4)
    cur_v_acc = round(v_correct / len_val, 4)
    cur_t_loss = round(tr_loss / len_train, 4)
    cur_v_loss = round(v_loss / len_val, 4)
    cur_inf_ms = round(np.mean(latencies) * 1000, 4)
    cur_time = round(time.time() - epoch_start, 2)

    # Log results
    history_hyb['epoch'].append(epoch + 1)
    history_hyb['train_loss'].append(cur_t_loss); history_hyb['train_acc'].append(cur_t_acc)
    history_hyb['val_loss'].append(cur_v_loss); history_hyb['val_acc'].append(cur_v_acc)
    history_hyb['inf_ms'].append(cur_inf_ms); history_hyb['time_sec'].append(cur_time)
    history_hyb['lr'].append(optimizer.param_groups[0]['lr'])

    print(f"{epoch+1:<8} | {cur_t_loss:<10.4f} | {cur_t_acc:<10.4f} | {cur_v_loss:<10.4f} | {cur_v_acc:<10.4f} | {cur_inf_ms:<10.4f} ms | {cur_time:<8}s")

    # --- [CRITICAL] IMPROVED SAVING & EARLY STOPPING LOGIC ---
    if cur_v_acc > best_hyb_acc:
        best_hyb_acc = cur_v_acc
        torch.save(model_hybrid.state_dict(), local_path)
        torch.save(model_hybrid.state_dict(), drive_path)
        print(f" ---> [SAVED] Model improved to {best_hyb_acc:.4f}. Checkpoint synced.")
        early_stop_counter = 0
    else:
        # Counter will only increase if accuracy is equal to or less than the best
        early_stop_counter += 1
        if early_stop_counter >= CONFIG["patience"]:
            print(f"\n[INFO] Early stopping triggered at epoch {epoch+1}")
            break

    scheduler.step()

# --- FINAL PERSISTENT LOGGING ---
pd.DataFrame(history_hyb).to_csv(os.path.join(EXPORT_DIR_HYB, "hybrid_training_history.csv"), index=False)
print(f"\n[FINISH] Hybrid Model Pipeline Successfully Completed.")
print(f"[RESULT] Peak Hybrid Validation Accuracy: {best_hyb_acc:.4f}")

# Memory management
del model_hybrid, optimizer; gc.collect(); torch.cuda.empty_cache()



# ==============================================================================
# PHASE 8: COMPREHENSIVE EVALUATION FOR PROPOSED HYBRID MODEL
# Artifacts: Confusion Matrix, Learning Curves, ROC-AUC, and Failure Analysis
# ==============================================================================

print("\n" + "="*80)
print("[INFO] Finalizing Research Artifacts for the Hybrid Model...")
print("="*80)


# ==============================================================================
# PHASE 5: COMPREHENSIVE EVALUATION (4-PASS TTA)
# ==============================================================================

print("\n" + "="*80 + "\n[INFO] Starting Multi-Pass Evaluation...\n" + "="*80)
eval_model = EnsembleStackingNet(BEST_3_NAMES).to(DEVICE)
if os.path.exists("temp_best_hybrid.pth"): eval_model.load_state_dict(torch.load("temp_best_hybrid.pth"))
eval_model.eval(); y_true, y_pred, y_score = [], [], []

with torch.no_grad():
    for imgs, lbls in hybrid_loaders['test']:
        imgs = imgs.to(DEVICE)
        # 4-Pass TTA
        o1 = eval_model(imgs)
        o2 = eval_model(torch.flip(imgs, dims=[3]))
        o3 = eval_model(torch.flip(imgs, dims=[2]))
        o4 = eval_model(torch.flip(imgs, dims=[2, 3]))
        averaged_logits = (o1 + o2 + o3 + o4) / 4
        probabilities = F.softmax(averaged_logits, dim=1); _, preds = torch.max(averaged_logits, 1)
        y_true.extend(lbls.numpy()); y_pred.extend(preds.cpu().numpy()); y_score.extend(probabilities.cpu().numpy())


# 9. CLASSIFICATION REPORT & METRICS EXPORT
# ------------------------------------------------------------------------------
report_hyb = classification_report(y_true, y_pred, target_names=hybrid_ds_raw.classes, output_dict=True, digits=4)
df_report_hyb = pd.DataFrame(report_hyb).transpose()
df_report_hyb.to_csv(os.path.join(EXPORT_DIR_HYB, "hybrid_final_metrics_report.csv"), float_format='%.4f')

print("\n[RESULT] Hybrid Model Test Accuracy: ", round(report_hyb['accuracy'] * 100, 2), "%")

# 10. CONFUSION MATRIX VISUALIZATION (HIGH RESOLUTION)
# ------------------------------------------------------------------------------

plt.figure(figsize=(18, 15))
cm_hyb = confusion_matrix(y_true, y_pred)
sns.heatmap(cm_hyb, annot=True, fmt='d', cmap='Greens',
            xticklabels=hybrid_ds_raw.classes, yticklabels=hybrid_ds_raw.classes)

plt.title('Confusion Matrix: Proposed Triple-Stream Hybrid', fontsize=20, fontweight='bold')
plt.ylabel('Ground Truth', fontsize=14); plt.xlabel('Predicted Class', fontsize=14)
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR_HYB, "hybrid_confusion_matrix.png"), dpi=400)
plt.close()

# 11. MULTI-CLASS ROC-AUC ANALYSIS
# ------------------------------------------------------------------------------
y_true_bin = label_binarize(y_true, classes=range(CONFIG["num_classes"]))
plt.figure(figsize=(12, 10))
for i in range(CONFIG["num_classes"]):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], np.array(y_score)[:, i])
    plt.plot(fpr, tpr, label=f'{hybrid_ds_raw.classes[i]} (AUC={auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.title('Multi-class ROC: Hybrid Model', fontsize=16)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.savefig(os.path.join(EXPORT_DIR_HYB, "hybrid_roc_auc.png"), dpi=300, bbox_inches='tight')
plt.close()

# 12. LEARNING CURVES (FROM SAVED LOGS)
# ------------------------------------------------------------------------------

# Read the CSV log we saved in Phase 13
df_hist_hyb = pd.read_csv(os.path.join(EXPORT_DIR_HYB, "hybrid_training_history.csv"))

plt.figure(figsize=(16, 6))
# Accuracy Curve
plt.subplot(1, 2, 1)
plt.plot(df_hist_hyb['train_acc'], label='Training Acc', marker='o', alpha=0.7)
plt.plot(df_hist_hyb['val_acc'], label='Validation Acc', marker='s', alpha=0.7)
plt.title('Hybrid Model: Accuracy Curve', fontweight='bold'); plt.legend(); plt.grid(True)

# Loss Curve
plt.subplot(1, 2, 2)
plt.plot(df_hist_hyb['train_loss'], label='Training Loss', marker='o', alpha=0.7)
plt.plot(df_hist_hyb['val_loss'], label='Validation Loss', marker='s', alpha=0.7)
plt.title('Hybrid Model: Loss Curve', fontweight='bold'); plt.legend(); plt.grid(True)

plt.savefig(os.path.join(EXPORT_DIR_HYB, "hybrid_learning_curves.png"), dpi=300)
plt.close()

# 13. HYBRID FAILURE ANALYSIS (IMAGE SAMPLES)
# ------------------------------------------------------------------------------
run_failure_analysis(eval_model, hybrid_loaders['test'], hybrid_ds_raw.classes, EXPORT_DIR_HYB)

print(f"\n[SUCCESS] All Hybrid Evaluation Artifacts Saved to: {EXPORT_DIR_HYB}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

------------------------------
[INFO] Loading Dataset and Partitioning...
------------------------------

Epoch    | Tr. Loss   | Tr. Acc    | Val Loss   | Val Acc    | Inference    | Time
---------------------------------------------------------------------------------------------------------
1        | 1.3486     | 0.7865     | 0.6213     | 0.9842     | 0.4736     ms | 108.78  s
 ---> [SAVED] Model improved to 0.9842. Checkpoint synced.
2        | 0.4648     | 0.9835     | 0.2716     | 0.9860     | 0.5129     ms | 112.22  s
 ---> [SAVED] Model improved to 0.9860. Checkpoint synced.
3        | 0.2647     | 0.9868     | 0.1931     | 0.9877     | 0.4786     ms | 115.89  s
 ---> [SAVED] Model improved to 0.9877. Checkpoint synced.
4        | 0.2001     | 0.9872     | 0.1380     | 0.9877     | 0.4722     ms | 114.25  s
5        | 0.1573     | 0.9891     | 0.126

In [ ]:
# ==============================================================================
# PHASE 8: SYSTEM IMPORTS & ENVIRONMENT SETUP
# ==============================================================================
import os
import time
import gc
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from google.colab import drive
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Mount Google Drive
drive.mount('/content/drive')

# Device Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Global Hyperparameters
CONFIG = {
    "batch_size": 32,
    "epochs": 50,
    "lr": 5e-6,
    "num_classes": 19,
    "patience": 8,
    "label_smoothing": 0.1,
    "weight_decay": 1e-2
}

# Project Paths
BASE_DIR = "/content/drive/MyDrive/BDMediHerb/Original Dataset"
PROJECT_ROOT = "/content/drive/MyDrive/BDMediHerb/Outputs"
BEST_3_NAMES = ["ResNet50", "ViT_B16"]


# ==============================================================================
# PHASE 1: UTILITY CLASSES & FUNCTIONS
# ==============================================================================

class MapDataset(torch.utils.data.Dataset):
    """Wraps a subset to apply transforms dynamically."""
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.dataset[index]
        if self.transform: x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.dataset)

def initialize_architecture(name):
    """Initializes backbone architectures to EXACTLY match the saved weights."""
    if name == "MobileNet_V3_Large":
        model = models.mobilenet_v3_large(weights="DEFAULT")
        num_ftrs = model.classifier[3].in_features
        # This matches your original training Phase 3 structure
        model.classifier[3] = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, CONFIG["num_classes"])
        )
    elif name == "ResNet50":
        model = models.resnet50(weights="DEFAULT")
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, CONFIG["num_classes"])
    elif name == "ViT_B16":
        model = models.vit_b_16(weights="DEFAULT")
        num_ftrs = model.heads.head.in_features
        model.heads.head = nn.Linear(num_ftrs, CONFIG["num_classes"])
    return model

class FocalLoss(nn.Module):
    """Focal Loss to handle hard-to-classify samples at terminal accuracy stages."""
    def __init__(self, alpha=1, gamma=2, label_smoothing=0.05):
        super(FocalLoss, self).__init__()
        self.alpha = alpha; self.gamma = gamma; self.label_smoothing = label_smoothing
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

def run_failure_analysis(model, loader, class_names, export_path):
    """Exports misclassified images and error logs for research audit."""
    model.eval()
    error_dir = os.path.join(export_path, "Failure_Analysis")
    os.makedirs(error_dir, exist_ok=True)
    error_log = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            preds = outputs.max(1)[1].cpu()
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    error_log.append({
                        "Actual": class_names[labels[i]],
                        "Predicted": class_names[preds[i]]
                    })

    # Save the log to the export_path
    pd.DataFrame(error_log).to_csv(os.path.join(export_path, "misclassifications.csv"), index=False)

    # FIX: Changed export_dir to error_dir
    print(f"[INFO] Failure analysis report saved to {error_dir}")

# ==============================================================================
# PHASE 2: DATA RE-LOADING & STRATIFIED PARTITIONING
# ==============================================================================

print("\n" + "-"*30 + "\n[INFO] Loading Dataset and Partitioning...\n" + "-"*30)
hybrid_ds_raw = datasets.ImageFolder(BASE_DIR)
hybrid_labels = np.array(hybrid_ds_raw.targets)
hybrid_indices = np.arange(len(hybrid_labels))

# Stratified Split 70:15:15
idx_train_val_hybrid, hybrid_test_idx = train_test_split(
    hybrid_indices, test_size=0.15, stratify=hybrid_labels, random_state=42
)
idx_train_hybrid, hybrid_val_idx = train_test_split(
    idx_train_val_hybrid, test_size=(0.15/0.85), stratify=hybrid_labels[idx_train_val_hybrid], random_state=42
)

def get_hybrid_loaders(train_idx, val_idx, test_idx, img_size, full_ds):
    """Generates resolution-aware loaders with heavy training augmentations."""
    train_trans = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    eval_trans = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return {
        'train': DataLoader(MapDataset(Subset(full_ds, train_idx), train_trans), batch_size=CONFIG["batch_size"], shuffle=True),
        'val': DataLoader(MapDataset(Subset(full_ds, val_idx), eval_trans), batch_size=CONFIG["batch_size"], shuffle=False),
        'test': DataLoader(MapDataset(Subset(full_ds, test_idx), eval_trans), batch_size=CONFIG["batch_size"], shuffle=False)
    }

# Initialize Loaders
HYBRID_RES = 224
hybrid_loaders = get_hybrid_loaders(idx_train_hybrid, hybrid_val_idx, hybrid_test_idx, HYBRID_RES, hybrid_ds_raw)

# ==============================================================================
# PHASE 3: COMPONENT ARCHITECTURE & ENSEMBLE CORE
# ==============================================================================

class FeatureExtractor(nn.Module):
    """Wraps backbones and loads individual weights, stripping classification heads."""
    def __init__(self, model_name):
        super(FeatureExtractor, self).__init__()
        base_model = initialize_architecture(model_name)
        weights_path = os.path.join(PROJECT_ROOT, model_name, "best_model.pth")
        base_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))

        if "ViT" in model_name:
            self.feat_dim = base_model.heads.head.in_features
            base_model.heads = nn.Identity()
        elif "ResNet" in model_name:
            self.feat_dim = base_model.fc.in_features
            base_model.fc = nn.Identity()
        elif "MobileNet" in model_name:
            self.feat_dim = base_model.classifier[0].in_features
            base_model.classifier = nn.Identity()
        self.backbone = base_model

    def forward(self, x):
        x = self.backbone(x)
        if len(x.shape) > 2: x = torch.flatten(x, 1)
        return x

class EnsembleStackingNet(nn.Module):
    """Meta-learner that dynamically adapts to the number of input models."""
    def __init__(self, model_names, num_classes=19):
        super(EnsembleStackingNet, self).__init__()
        # Dynamic streams using ModuleList
        self.streams = nn.ModuleList([FeatureExtractor(name) for name in model_names])
        self.heads = nn.ModuleList([nn.Linear(stream.feat_dim, num_classes) for stream in self.streams])

        # Meta-learner: input size depends on the number of models
        self.meta_learner = nn.Sequential(
            nn.Linear(num_classes * len(model_names), 256),
            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        logits_list = []
        for i, stream in enumerate(self.streams):
            features = stream(x)
            logits = self.heads[i](features)
            logits_list.append(logits)

        stacked_logits = torch.cat(logits_list, dim=-1)
        return self.meta_learner(stacked_logits)


# ==============================================================================
# PHASE 4: TRAINING EXECUTION
# ==============================================================================

model_hybrid = EnsembleStackingNet(BEST_3_NAMES).to(DEVICE)
param_groups = []
for stream in model_hybrid.streams:
    param_groups.append({'params': stream.parameters(), 'lr': 1e-6})
for head in model_hybrid.heads:
    param_groups.append({'params': head.parameters(), 'lr': 1e-4})
param_groups.append({'params': model_hybrid.meta_learner.parameters(), 'lr': 1e-4})
optimizer = optim.AdamW(param_groups, weight_decay=0.01)
criterion = FocalLoss(gamma=2, label_smoothing=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)


# ------------------------------------------------------------------------------
# 5. EXECUTION: HIGH-PRECISION TRAINING & LATENCY TRACKING
# ------------------------------------------------------------------------------
best_hyb_acc = 0.0
early_stop_counter = 0
CONFIG["patience"] = 8

# Archive indices lengths for precise accuracy calculation
len_train = len(idx_train_hybrid)
len_val = len(hybrid_val_idx)

# Paths for saving
EXPORT_DIR_HYB = os.path.join(PROJECT_ROOT, "Hybrid_DualStreamRV")
os.makedirs(EXPORT_DIR_HYB, exist_ok=True)
local_path = "temp_best_hybrid.pth"
drive_path = os.path.join(EXPORT_DIR_HYB, "best_hybrid_model.pth")

# Initialize history dictionary before starting the loop
history_hyb = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'inf_ms': [], 'time_sec': [], 'lr': []}

print(f"\n{'Epoch':<8} | {'Tr. Loss':<10} | {'Tr. Acc':<10} | {'Val Loss':<10} | {'Val Acc':<10} | {'Inference':<12} | {'Time'}")
print("-" * 105)


# Now your existing loop starts...
for epoch in range(CONFIG["epochs"]):
    epoch_start = time.time()

    # --- TRAINING PHASE ---
    model_hybrid.train()
    tr_loss, tr_correct = 0.0, 0
    for imgs, lbls in hybrid_loaders['train']:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model_hybrid(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * imgs.size(0)
        tr_correct += (out.max(1)[1] == lbls).sum().item()

    # --- VALIDATION PHASE ---
    model_hybrid.eval()
    v_loss, v_correct, latencies = 0.0, 0, []
    with torch.no_grad():
        for imgs, lbls in hybrid_loaders['val']:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            tic = time.time()
            out = model_hybrid(imgs)
            latencies.append((time.time() - tic) / imgs.size(0))
            v_loss += criterion(out, lbls).item() * imgs.size(0)
            v_correct += (out.max(1)[1] == lbls).sum().item()

    # --- METRICS CALCULATION ---
    cur_t_acc = round(tr_correct / len_train, 4)
    cur_v_acc = round(v_correct / len_val, 4)
    cur_t_loss = round(tr_loss / len_train, 4)
    cur_v_loss = round(v_loss / len_val, 4)
    cur_inf_ms = round(np.mean(latencies) * 1000, 4)
    cur_time = round(time.time() - epoch_start, 2)

    # Log results
    history_hyb['epoch'].append(epoch + 1)
    history_hyb['train_loss'].append(cur_t_loss); history_hyb['train_acc'].append(cur_t_acc)
    history_hyb['val_loss'].append(cur_v_loss); history_hyb['val_acc'].append(cur_v_acc)
    history_hyb['inf_ms'].append(cur_inf_ms); history_hyb['time_sec'].append(cur_time)
    history_hyb['lr'].append(optimizer.param_groups[0]['lr'])

    print(f"{epoch+1:<8} | {cur_t_loss:<10.4f} | {cur_t_acc:<10.4f} | {cur_v_loss:<10.4f} | {cur_v_acc:<10.4f} | {cur_inf_ms:<10.4f} ms | {cur_time:<8}s")

    # --- [CRITICAL] IMPROVED SAVING & EARLY STOPPING LOGIC ---
    if cur_v_acc > best_hyb_acc:
        best_hyb_acc = cur_v_acc
        torch.save(model_hybrid.state_dict(), local_path)
        torch.save(model_hybrid.state_dict(), drive_path)
        print(f" ---> [SAVED] Model improved to {best_hyb_acc:.4f}. Checkpoint synced.")
        early_stop_counter = 0
    else:
        # Counter will only increase if accuracy is equal to or less than the best
        early_stop_counter += 1
        if early_stop_counter >= CONFIG["patience"]:
            print(f"\n[INFO] Early stopping triggered at epoch {epoch+1}")
            break

    scheduler.step()

# --- FINAL PERSISTENT LOGGING ---
pd.DataFrame(history_hyb).to_csv(os.path.join(EXPORT_DIR_HYB, "hybrid_training_history.csv"), index=False)
print(f"\n[FINISH] Hybrid Model Pipeline Successfully Completed.")
print(f"[RESULT] Peak Hybrid Validation Accuracy: {best_hyb_acc:.4f}")

# Memory management
del model_hybrid, optimizer; gc.collect(); torch.cuda.empty_cache()



# ==============================================================================
# PHASE 8: COMPREHENSIVE EVALUATION FOR PROPOSED HYBRID MODEL
# Artifacts: Confusion Matrix, Learning Curves, ROC-AUC, and Failure Analysis
# ==============================================================================

print("\n" + "="*80)
print("[INFO] Finalizing Research Artifacts for the Hybrid Model...")
print("="*80)


# ==============================================================================
# PHASE 5: COMPREHENSIVE EVALUATION (4-PASS TTA)
# ==============================================================================

print("\n" + "="*80 + "\n[INFO] Starting Multi-Pass Evaluation...\n" + "="*80)
eval_model = EnsembleStackingNet(BEST_3_NAMES).to(DEVICE)
if os.path.exists("temp_best_hybrid.pth"): eval_model.load_state_dict(torch.load("temp_best_hybrid.pth"))
eval_model.eval(); y_true, y_pred, y_score = [], [], []

with torch.no_grad():
    for imgs, lbls in hybrid_loaders['test']:
        imgs = imgs.to(DEVICE)
        # 4-Pass TTA
        o1 = eval_model(imgs)
        o2 = eval_model(torch.flip(imgs, dims=[3]))
        o3 = eval_model(torch.flip(imgs, dims=[2]))
        o4 = eval_model(torch.flip(imgs, dims=[2, 3]))
        averaged_logits = (o1 + o2 + o3 + o4) / 4
        probabilities = F.softmax(averaged_logits, dim=1); _, preds = torch.max(averaged_logits, 1)
        y_true.extend(lbls.numpy()); y_pred.extend(preds.cpu().numpy()); y_score.extend(probabilities.cpu().numpy())


# 9. CLASSIFICATION REPORT & METRICS EXPORT
# ------------------------------------------------------------------------------
report_hyb = classification_report(y_true, y_pred, target_names=hybrid_ds_raw.classes, output_dict=True, digits=4)
df_report_hyb = pd.DataFrame(report_hyb).transpose()
df_report_hyb.to_csv(os.path.join(EXPORT_DIR_HYB, "hybrid_final_metrics_report.csv"), float_format='%.4f')

print("\n[RESULT] Hybrid Model Test Accuracy: ", round(report_hyb['accuracy'] * 100, 2), "%")

# 10. CONFUSION MATRIX VISUALIZATION (HIGH RESOLUTION)
# ------------------------------------------------------------------------------

plt.figure(figsize=(18, 15))
cm_hyb = confusion_matrix(y_true, y_pred)
sns.heatmap(cm_hyb, annot=True, fmt='d', cmap='Greens',
            xticklabels=hybrid_ds_raw.classes, yticklabels=hybrid_ds_raw.classes)

plt.title('Confusion Matrix: Proposed Triple-Stream Hybrid', fontsize=20, fontweight='bold')
plt.ylabel('Ground Truth', fontsize=14); plt.xlabel('Predicted Class', fontsize=14)
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR_HYB, "hybrid_confusion_matrix.png"), dpi=400)
plt.close()

# 11. MULTI-CLASS ROC-AUC ANALYSIS
# ------------------------------------------------------------------------------
y_true_bin = label_binarize(y_true, classes=range(CONFIG["num_classes"]))
plt.figure(figsize=(12, 10))
for i in range(CONFIG["num_classes"]):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], np.array(y_score)[:, i])
    plt.plot(fpr, tpr, label=f'{hybrid_ds_raw.classes[i]} (AUC={auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.title('Multi-class ROC: Hybrid Model', fontsize=16)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.savefig(os.path.join(EXPORT_DIR_HYB, "hybrid_roc_auc.png"), dpi=300, bbox_inches='tight')
plt.close()

# 12. LEARNING CURVES (FROM SAVED LOGS)
# ------------------------------------------------------------------------------

# Read the CSV log we saved in Phase 13
df_hist_hyb = pd.read_csv(os.path.join(EXPORT_DIR_HYB, "hybrid_training_history.csv"))

plt.figure(figsize=(16, 6))
# Accuracy Curve
plt.subplot(1, 2, 1)
plt.plot(df_hist_hyb['train_acc'], label='Training Acc', marker='o', alpha=0.7)
plt.plot(df_hist_hyb['val_acc'], label='Validation Acc', marker='s', alpha=0.7)
plt.title('Hybrid Model: Accuracy Curve', fontweight='bold'); plt.legend(); plt.grid(True)

# Loss Curve
plt.subplot(1, 2, 2)
plt.plot(df_hist_hyb['train_loss'], label='Training Loss', marker='o', alpha=0.7)
plt.plot(df_hist_hyb['val_loss'], label='Validation Loss', marker='s', alpha=0.7)
plt.title('Hybrid Model: Loss Curve', fontweight='bold'); plt.legend(); plt.grid(True)

plt.savefig(os.path.join(EXPORT_DIR_HYB, "hybrid_learning_curves.png"), dpi=300)
plt.close()

# 13. HYBRID FAILURE ANALYSIS (IMAGE SAMPLES)
# ------------------------------------------------------------------------------
run_failure_analysis(eval_model, hybrid_loaders['test'], hybrid_ds_raw.classes, EXPORT_DIR_HYB)

print(f"\n[SUCCESS] All Hybrid Evaluation Artifacts Saved to: {EXPORT_DIR_HYB}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

------------------------------
[INFO] Loading Dataset and Partitioning...
------------------------------

Epoch    | Tr. Loss   | Tr. Acc    | Val Loss   | Val Acc    | Inference    | Time
---------------------------------------------------------------------------------------------------------
1        | 1.4007     | 0.7474     | 0.5750     | 0.9737     | 0.4606     ms | 206.47  s
 ---> [SAVED] Model improved to 0.9737. Checkpoint synced.
2        | 0.4751     | 0.9887     | 0.2809     | 0.9860     | 0.5009     ms | 216.36  s
 ---> [SAVED] Model improved to 0.9860. Checkpoint synced.
3        | 0.2503     | 0.9914     | 0.1601     | 0.9860     | 0.4893     ms | 217.48  s
4        | 0.1726     | 0.9902     | 0.1203     | 0.9877     | 0.4484     ms | 206.44  s
 ---> [SAVED] Model improved to 0.9877. Checkpoint synced.
5        | 0.1420     | 0.9921     | 0.109